# GRPO (Group Relative Policy Optimization) Loss

**难度:** Hard

实现 GRPO（组相对策略优化）损失。

GRPO 在每个提示组内归一化奖励以计算优势值，然后使用这些组相对优势优化策略。

**签名:** `grpo_loss(logps, rewards, group_ids, eps=1e-5) -> Tensor`

**参数:**
- `logps` — 策略对数概率 (B,)
- `rewards` — 标量奖励 (B,)
- `group_ids` — 整数组标识符 (B,)
- `eps` — 数值稳定性的 epsilon

**返回:** 标量损失

**约束:**
- 组内 z-score 归一化：`A_i = (r_i - mean_g) / (std_g + eps)`
- 优势值必须 detach（梯度仅通过 logps 流动）
- 损失 = `-mean(A_i * logps_i)`

**提示:**
1. 对每个组 g：A_i = (r_i - mean_g) / (std_g + eps)
2. advantages = A.detach()（梯度不流经奖励）
3. loss = -(advantages * logps).mean()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
# ✅ SOLUTION

def grpo_loss(logps: Tensor, rewards: Tensor, group_ids: Tensor,
              eps: float = 1e-5) -> Tensor:
    # ---- Step 1: 组内 z-score 归一化 ----
    # 对每个组（同一 prompt 的多个回复），将奖励标准化为优势值
    # A_i = (r_i - mean_g) / (std_g + eps)
    # 直觉：正优势 → 这个回复比同组平均好，应该增加其概率
    #       负优势 → 比平均差，应该降低其概率
    unique_ids = group_ids.unique()
    advantages = torch.empty_like(rewards)

    for gid in unique_ids:
        mask = group_ids == gid   # 选出属于这个组的所有样本
        r_g = rewards[mask]       # 该组的奖励
        mean_g = r_g.mean()       # 组内均值
        std_g = r_g.std(unbiased=False)  # 组内标准差（无偏=False，用 N 而非 N-1）
        advantages[mask] = (r_g - mean_g) / (std_g + eps)

    # ---- Step 2: 截断梯度 ----
    # advantages 来自奖励（外部信号），不应该有梯度
    # 梯度只应该通过 logps 流动，告诉模型"增加/减少这个 token 的概率"
    advantages_detached = advantages.detach()

    # ---- Step 3: 策略梯度损失 ----
    # loss = -E[A_i * log π(a_i)]
    # 正 A * 正 logp → 负贡献 → loss 减小（好回复概率增加）
    # 负 A * 正 logp → 正贡献 → loss 增大（差回复概率减少）
    return -(advantages_detached * logps).mean()

In [ ]:
# Demo
logps = torch.tensor([0.0, -0.5, -1.0, -1.5])
rewards = torch.tensor([1.0, 0.8, 0.2, 0.0])
group_ids = torch.tensor([0, 0, 1, 1])
print('Loss:', grpo_loss(logps, rewards, group_ids).item())

In [ ]:
from torch_judge import check
check('grpo_loss')

## 📝 核心思路总结

1. **GRPO = 组内相对优势策略优化**：同一 prompt 的多个回复互相比较，而非跨 prompt
2. **z-score 归一化消除组间奖励尺度差异**：不同 prompt 的奖励可能差异很大，归一化后统一尺度
3. **advantages.detach() 是关键**：梯度只通过 logps 流动，奖励被视为常量